# 03: Model C — LLaMA-3.2-1B-Instruct 4-bit QLoRA Fine-Tuning
**Course**: Foundations of Generative AI (ICT4403) - Assignment 3  
**Architecture**: Decoder-Only Autoregressive Causal LM (`meta-llama/Llama-3.2-1B-Instruct`)  
**Adaptation**: 4-bit NormalFloat4 (NF4) QLoRA with rank $r=16$, $\alpha=32$ on all linear projections  
**Hardware Target**: Google Colab T4 GPU (16 GB VRAM)

### Key Technical Features
1. **Zero-Shot Baseline**: Evaluates un-adapted instruction model on the 1,000-sentence test set.
2. **Response-Only Loss Masking**: Cross-entropy is computed solely on Hindi tokens (prompt tokens masked with `-100`).
3. **Left-Padded Batched Inference**: Aligns sequence generation heads across variable prompt lengths.
4. **Full Resumability**: Saves step checkpoints and cumulative training time to Google Drive.

## 1. Environment Setup, Colab Secrets & Drive Mount

In [ ]:
import os

drive_metrics_path = "/content/drive/MyDrive/mt-fgai-assignment3/src/metrics.py"

full_metrics_code = '''"""Evaluation metrics, parameter accounting, and timing benchmarks for Machine Translation."""
import json
import logging
import os
import time
from typing import Any, Dict, List, Optional, Tuple, Union

import pandas as pd
import sacrebleu
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

def compute_sacrebleu(hypotheses: List[str], references: List[str]) -> Tuple[float, str]:
    """Compute corpus-level BLEU score with standard 13a tokenization and return full signature."""
    metric = sacrebleu.metrics.BLEU(tokenize="13a")
    score = metric.corpus_score(hypotheses, [references])
    try:
        sig = str(metric.get_signature())
    except Exception:
        sig = "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp"
    return round(float(score.score), 2), sig

def compute_chrf(hypotheses: List[str], references: List[str]) -> Tuple[float, float, str]:
    """Compute corpus chrF (character n-grams) and chrF++ (character + word n-grams)."""
    metric_chrf = sacrebleu.metrics.CHRF()
    score_chrf = metric_chrf.corpus_score(hypotheses, [references])
    try:
        sig = str(metric_chrf.get_signature())
    except Exception:
        sig = "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no"

    metric_pp = sacrebleu.metrics.CHRF(word_order=2)
    score_pp = metric_pp.corpus_score(hypotheses, [references])
    return round(float(score_chrf.score), 2), round(float(score_pp.score), 2), sig

def count_parameters(model: torch.nn.Module) -> Dict[str, int]:
    """Calculate the total and trainable parameter counts of any PyTorch model."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "frozen_params": total_params - trainable_params,
    }

class CumulativeTimer:
    """Timer supporting cumulative elapsed duration across Google Colab session disconnects."""
    def __init__(self, initial_seconds: float = 0.0):
        self.cumulative_seconds = float(initial_seconds)
        self.start_time: Optional[float] = None
        self.is_running = False

    def start(self) -> None:
        if not self.is_running:
            self.start_time = time.perf_counter()
            self.is_running = True

    def stop(self) -> float:
        if self.is_running and self.start_time is not None:
            elapsed = time.perf_counter() - self.start_time
            self.cumulative_seconds += elapsed
            self.is_running = False
            self.start_time = None
        return self.cumulative_seconds

    def elapsed(self) -> float:
        total = self.cumulative_seconds
        if self.is_running and self.start_time is not None:
            total += time.perf_counter() - self.start_time
        return total

def get_gpu_device_name() -> str:
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "CPU"

def save_predictions_csv(ids, sources, references, hypotheses, output_path: str) -> None:
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df = pd.DataFrame({"id": ids, "source": sources, "reference": references, "hypothesis": hypotheses})
    df.to_csv(output_path, index=False, encoding="utf-8")

def save_metrics_json(
    model_name, architecture_type, adaptation_method, total_params, trainable_params,
    train_data_size, train_time_sec, inference_time_total_sec, num_test_samples,
    bleu, bleu_signature, chrf, chrf_pp, chrf_signature, decoding_settings,
    batch_size, hyperparameters, output_path, cleaned_outputs_count=0,
) -> Dict[str, Any]:
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    inference_per_sentence = round(inference_time_total_sec / num_test_samples, 6) if num_test_samples > 0 else 0.0
    payload = {
        "model_name": model_name,
        "architecture_type": architecture_type,
        "adaptation_method": adaptation_method,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "train_data_size": train_data_size,
        "train_time_sec": round(train_time_sec, 2),
        "inference_time_total_sec": round(inference_time_total_sec, 3),
        "inference_time_per_sentence_sec": inference_per_sentence,
        "inference_time_per_sentence_ms": round(inference_per_sentence * 1000.0, 2),
        "num_test_samples": num_test_samples,
        "bleu": bleu,
        "bleu_signature": bleu_signature,
        "chrf": chrf,
        "chrf_pp": chrf_pp,
        "chrf_signature": chrf_signature,
        "batch_size": batch_size,
        "decoding_settings": decoding_settings,
        "cleaned_outputs_count": cleaned_outputs_count,
        "gpu": get_gpu_device_name(),
        "hyperparameters": hyperparameters,
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return payload
'''

with open(drive_metrics_path, "w", encoding="utf-8") as f:
    f.write(full_metrics_code)

# Force Python to reload the updated module in memory
import sys
import importlib
if "src.metrics" in sys.modules:
    importlib.reload(sys.modules["src.metrics"])
    print("✅ src.metrics successfully reloaded in Colab memory!")
else:
    print("✅ Full src/metrics.py restored to Google Drive!")

✅ src.metrics successfully reloaded in Colab memory!


In [ ]:
import os
import sys

DRIVE_DIR = "/content/drive/MyDrive/mt-fgai-assignment3"

try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Google Drive ready at: {DRIVE_DIR}")
except ImportError:
    print("Running locally.")
    DRIVE_DIR = "."

if os.path.exists(DRIVE_DIR) and os.path.exists(os.path.join(DRIVE_DIR, "configs", "config.yaml")):
    REPO_ROOT = DRIVE_DIR
    os.chdir(REPO_ROOT)
    print(f"Active project root set to Google Drive: {REPO_ROOT}")
else:
    REPO_ROOT = os.path.abspath(".")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import google.colab
    !pip install -q transformers>=4.45.0 accelerate peft trl bitsandbytes sentencepiece sacrebleu evaluate datasets pyyaml
except ImportError:
    pass

# Read Hugging Face token from Colab Secrets
hf_token = os.environ.get("HF_TOKEN", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Successfully authenticated using Colab Secret HF_TOKEN.")
except Exception:
    if hf_token:
        print("Using HF_TOKEN from environment variables.")
    else:
        print("Warning: HF_TOKEN not found. If downloading gated models, ensure authentication.")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


Mounting Google Drive...
Mounted at /content/drive
Google Drive ready at: /content/drive/MyDrive/mt-fgai-assignment3
Active project root set to Google Drive: /content/drive/MyDrive/mt-fgai-assignment3
Successfully authenticated using Colab Secret HF_TOKEN.
Active Device: cuda (Tesla T4)


## 2. Load Configuration and Splits

In [ ]:
import yaml
from src.data import load_split_from_jsonl

with open(os.path.join(REPO_ROOT, "configs", "config.yaml"), "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

splits_dir = os.path.join(REPO_ROOT, config["paths"]["splits_dir"])
train_data = load_split_from_jsonl("train", splits_dir=splits_dir)
val_data = load_split_from_jsonl("val", splits_dir=splits_dir)
test_data = load_split_from_jsonl("test", splits_dir=splits_dir)

# Select configured training subset (e.g., 20,000 pairs) for manageable LLM fine-tuning
subset_size = min(2500, config["splits"].get("llm_subset_size", 2500))
train_subset = train_data[:subset_size]
print(f"Selected {len(train_subset):,} pairs for fast QLoRA fine-tuning.")

Selected 2,500 pairs for fast QLoRA fine-tuning.


## 3. Load LLaMA-3.2-1B-Instruct in 4-bit NF4 Quantization

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.metrics import count_parameters

model_id = config["models"]["llama_3_2_1b"]["name"]
print(f"Loading 4-bit quantized base model: {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Essential for batched generation

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

base_params = count_parameters(model)
print(f"Base Model Total Parameters: {base_params['total_params']:,}")
print(f"Base Model Trainable Parameters: {base_params['trainable_params']:,}")

Loading 4-bit quantized base model: meta-llama/Llama-3.2-1B-Instruct...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Base Model Total Parameters: 749,275,136
Base Model Trainable Parameters: 262,735,872


## 4. Zero-Shot Baseline Evaluation (1,000 Test Sentences)
Quantifies out-of-the-box translation capability before any parameter adaptation.

In [ ]:
from src.prompts import format_prompt_for_inference
from src.generation import generate_hf_causal_lm
from src.metrics import compute_sacrebleu, compute_chrf, save_predictions_csv, save_metrics_json

zeroshot_metric_path = "results/metrics/metrics_llama_zeroshot.json"
zeroshot_pred_path = "results/predictions/predictions_llama_zeroshot.csv"

test_prompts = [format_prompt_for_inference(x["en"], tokenizer) for x in test_data]
all_references = [x["hi"] for x in test_data]
all_sources = [x["en"] for x in test_data]
all_ids = [x["id"] for x in test_data]

if os.path.exists(zeroshot_metric_path):
    print(f"Zero-shot baseline already evaluated: {zeroshot_metric_path}")
else:
    print("Evaluating zero-shot baseline on 1,000 test sentences...")
    zs_hypotheses, zs_time, zs_cleaned = generate_hf_causal_lm(
        model=model,
        tokenizer=tokenizer,
        prompts=test_prompts,
        batch_size=8,
        max_new_tokens=config["generation"]["max_new_tokens"],
        temperature=0.0,
    )
    zs_bleu, zs_bleu_sig = compute_sacrebleu(zs_hypotheses, all_references)
    zs_chrf, zs_chrf_pp, zs_chrf_sig = compute_chrf(zs_hypotheses, all_references)

    print(f"Zero-shot SacreBLEU: {zs_bleu} | chrF: {zs_chrf} | chrF++: {zs_chrf_pp}")
    save_predictions_csv(all_ids, all_sources, all_references, zs_hypotheses, zeroshot_pred_path)
    save_metrics_json(
        model_name="llama_3_2_1b_zeroshot",
        architecture_type="decoder_only",
        adaptation_method="zero_shot",
        total_params=base_params["total_params"],
        trainable_params=0,
        train_data_size=0,
        train_time_sec=0.0,
        inference_time_total_sec=zs_time,
        num_test_samples=len(test_data),
        bleu=zs_bleu,
        bleu_signature=zs_bleu_sig,
        chrf=zs_chrf,
        chrf_pp=zs_chrf_pp,
        chrf_signature=zs_chrf_sig,
        decoding_settings={"strategy": "greedy", "max_new_tokens": config["generation"]["max_new_tokens"]},
        batch_size=8,
        hyperparameters={"model": model_id},
        output_path=zeroshot_metric_path,
        cleaned_outputs_count=zs_cleaned,
    )

Evaluating zero-shot baseline on 1,000 test sentences...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Zero-shot SacreBLEU: 7.11 | chrF: 28.05 | chrF++: 26.33


## 5. Configure QLoRA Adapters & Masked Response Training

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from src.prompts import ResponseOnlyDataCollator

model = prepare_model_for_kbit_training(model)
llama_cfg = config["models"]["llama_3_2_1b"]

peft_config = LoraConfig(
    r=llama_cfg["lora_r"],
    lora_alpha=llama_cfg["lora_alpha"],
    lora_dropout=llama_cfg["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=llama_cfg["target_modules"],
)

peft_model = get_peft_model(model, peft_config)
peft_params = count_parameters(peft_model)
print(f"QLoRA Model Total Parameters: {peft_params['total_params']:,}")
print(f"QLoRA Trainable Adapter Parameters: {peft_params['trainable_params']:,} ({peft_params['trainable_params']/peft_params['total_params']*100:.2f}% of total)")

QLoRA Model Total Parameters: 760,547,328
QLoRA Trainable Adapter Parameters: 11,272,192 (1.48% of total)


## 6. Resumable Fine-Tuning Loop (Drive Checkpointing & Cumulative Timer)

In [ ]:
import json
import os
from datasets import Dataset
from transformers import Trainer, TrainingArguments, TrainerCallback, DataCollatorForSeq2Seq
from src.metrics import CumulativeTimer

output_dir = os.path.join(DRIVE_DIR, "llama_qlora_checkpoints")
os.makedirs(output_dir, exist_ok=True)
timer_file = os.path.join(output_dir, "cumulative_time.json")

init_time = 0.0
if os.path.exists(timer_file):
    with open(timer_file, "r") as f:
        init_time = json.load(f).get("cumulative_seconds", 0.0)
    print(f"Restored previous cumulative training time: {init_time:.1f}s")

timer = CumulativeTimer(initial_seconds=init_time)
timer.start()

class CumulativeTimeCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        with open(timer_file, "w") as f:
            json.dump({"cumulative_seconds": timer.elapsed(), "step": state.global_step}, f)

# 1. Pre-tokenize the 2,500 training pairs with response-only masking (-100 on prompt tokens)
print(f"Preparing {len(train_subset):,} pairs with prompt masking (-100)...")
formatted_data = []
for item in train_subset:
    src = item["en"]
    tgt = item["hi"]
    prompt_msgs = [{"role": "user", "content": f"Translate the following English sentence to Hindi. Output only the Hindi translation.\n\nEnglish: {src}"}]
    prompt_str = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)

    full_msgs = [{"role": "user", "content": f"Translate the following English sentence to Hindi. Output only the Hindi translation.\n\nEnglish: {src}"}, {"role": "assistant", "content": tgt}]
    full_str = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer.encode(full_str, add_special_tokens=False)
    if not full_ids or full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(tokenizer.eos_token_id)

    full_ids = full_ids[:256]
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    formatted_data.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    })

train_dataset = Dataset.from_list(formatted_data)

# 2. Training Arguments with remove_unused_columns=False and warmup_steps
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    learning_rate=llama_cfg["lr"],
    per_device_train_batch_size=llama_cfg["batch_size"],
    gradient_accumulation_steps=llama_cfg["gradient_accumulation_steps"],
    num_train_epochs=llama_cfg["epochs"],
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=25,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[CumulativeTimeCallback()],
)

has_checkpoint = any(d.startswith("checkpoint-") for d in os.listdir(output_dir))
print(f"Resume from checkpoint available: {has_checkpoint}")
print(f"Training on {len(train_dataset):,} pairs (~{len(train_dataset) // (llama_cfg['batch_size'] * llama_cfg['gradient_accumulation_steps'])} gradient steps)...")

trainer.train(resume_from_checkpoint=has_checkpoint)
total_train_time = timer.stop()

# Save final LoRA adapter to Google Drive
adapter_dir = os.path.join(DRIVE_DIR, "lora_llama32_hindi")
peft_model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
with open(timer_file, "w") as f:
    json.dump({"cumulative_seconds": total_train_time}, f)
print(f"\nQLoRA training complete! Total time: {total_train_time:.1f}s. Adapter saved to: {adapter_dir}")

Preparing 2,500 pairs with prompt masking (-100)...
Resume from checkpoint available: False
Training on 2,500 pairs (~78 gradient steps)...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,1.790398
50,1.633050
75,1.587296



QLoRA training complete! Total time: 332.3s. Adapter saved to: /content/drive/MyDrive/mt-fgai-assignment3/lora_llama32_hindi


## 7. Fine-Tuned Model Test Evaluation (1,000 Sentences)
Evaluates the adapted LLaMA model using left-padded greedy decoding on the identical test set.

In [ ]:
import sacrebleu
from src.generation import generate_hf_causal_lm
from src.metrics import save_predictions_csv, save_metrics_json

print("Evaluating fine-tuned LLaMA-3.2 on 1,000 test sentences...")
hypotheses, inf_time, cleaned_count = generate_hf_causal_lm(
    model=peft_model,
    tokenizer=tokenizer,
    prompts=test_prompts,
    batch_size=8,
    max_new_tokens=config["generation"]["max_new_tokens"],
    temperature=0.0,
)

# Robust SacreBLEU & chrF++ calculation
bleu_obj = sacrebleu.corpus_bleu(hypotheses, [all_references], tokenize="13a")
bleu_score = round(float(bleu_obj.score), 2)
bleu_sig = str(getattr(bleu_obj, "signature", "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp"))

chrf_obj = sacrebleu.corpus_chrf(hypotheses, [all_references])
chrf_pp_obj = sacrebleu.corpus_chrf(hypotheses, [all_references], word_order=2)
chrf_score = round(float(chrf_obj.score), 2)
chrf_pp_score = round(float(chrf_pp_obj.score), 2)
chrf_sig = str(getattr(chrf_obj, "signature", "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no"))

print(f"\n=== Model C (LLaMA-3.2-1B QLoRA) Test Evaluation ===")
print(f"SacreBLEU: {bleu_score} (Signature: {bleu_sig})")
print(f"chrF     : {chrf_score}")
print(f"chrF++   : {chrf_pp_score}")
print(f"Inference Latency: {inf_time / len(test_data) * 1000.0:.2f} ms/sentence")
print(f"Cleaned Outputs Count: {cleaned_count}")

pred_path = "results/predictions/predictions_llama_qlora.csv"
metric_path = "results/metrics/metrics_llama_qlora.json"

save_predictions_csv(all_ids, all_sources, all_references, hypotheses, pred_path)
save_metrics_json(
    model_name="llama_3_2_1b_qlora",
    architecture_type="decoder_only",
    adaptation_method="qlora_4bit",
    total_params=peft_params["total_params"],
    trainable_params=peft_params["trainable_params"],
    train_data_size=len(train_subset),
    train_time_sec=total_train_time,
    inference_time_total_sec=inf_time,
    num_test_samples=len(test_data),
    bleu=bleu_score,
    bleu_signature=bleu_sig,
    chrf=chrf_score,
    chrf_pp=chrf_pp_score,
    chrf_signature=chrf_sig,
    decoding_settings={"strategy": "greedy", "max_new_tokens": config["generation"]["max_new_tokens"]},
    batch_size=8,
    hyperparameters=llama_cfg,
    output_path=metric_path,
    cleaned_outputs_count=cleaned_count,
)

print("\n--- Sample Test Predictions ---")
for i in range(5):
    print(f"\n[Test Sample {i+1}]")
    print(f"  Source    : {all_sources[i]}")
    print(f"  Reference : {all_references[i]}")
    print(f"  Hypothesis: {hypotheses[i]}")

Evaluating fine-tuned LLaMA-3.2 on 1,000 test sentences...

=== Model C (LLaMA-3.2-1B QLoRA) Test Evaluation ===
SacreBLEU: 8.11 (Signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp)
chrF     : 32.26
chrF++   : 30.16
Inference Latency: 749.98 ms/sentence
Cleaned Outputs Count: 0

--- Sample Test Predictions ---

[Test Sample 1]
  Source    : The US warns its own companies about economic espionage by other countries.
  Reference : अमेरिका ने अन्य देशों द्वारा अपनी ही कंपनियों की आर्थिक जासूसी के बारे में चेतावनी दी है।
  Hypothesis: अमेरिकी कंपनियों को दूसरे देशों द्वारा आर्थिक छेड़छाड़ के बारे में सूचित किया गया है।

[Test Sample 2]
  Source    : Most of the skirmishes have taken place in Sofala province, which is several hundred kilometres north of Maputo, but hosts Beira, the port that miners, including Rio Tinto and Vale, rely on to export coal.
  Reference : झड़पों के अधिकांशसोफा ला प्रांत मेंहै, जोमापुटो के उत्तर में कई सौ किलोमीटर की दूरी पर है, जिसमें मेजबान बीरा पोर्ट है जो